# On-Device AI 시험 대비 연습 문제

각 문제에서 `## Write Answer` 부분을 채워 넣으세요.

**출제 범위**
- Part 1. Pruning (문제 1-7)
- Part 2. Quantization (문제 8-13)
- Part 3. Knowledge Distillation (문제 14-16)

# Part 1. Pruning

가중치의 중요도(importance)를 판단하는 기준을 이해하고, 이를 통해 불필요한 연산을 제거하는 과정을 다루는 문제입니다.

## 문제 1. Fine-grained Magnitude Pruning

[문제]
Q 1) 가중치 텐서의 절댓값을 기준으로 지정된 희소도(sparsity)만큼 0으로 마스킹하는 Fine-grained Magnitude Pruning을 구현합니다.

[요구사항]
- 희소도에 해당하는 절댓값 하위 임계값(threshold)을 torch.kthvalue로 계산
- 임계값 미만의 가중치를 0으로 마스킹한 텐서 반환

In [ ]:
# [코드]
def prune_weight_fine_grained(weight: torch.Tensor, sparsity: float) -> torch.Tensor:
    sparsity = min(1.0, max(0.0, sparsity))
    if sparsity == 1.0:
        weight.zero_()
        return torch.zeros_like(weight)
    elif sparsity == 0.0:
        return torch.ones_like(weight)

    num_pruned_elements = ### 작성 필요 ###
    importance = ### 작성 필요 ###
    threshold = ### 작성 필요 ###
    mask = ### 작성 필요 ###

    weight.mul_(mask)
    return mask

## 문제 2. Channel-level Pruning

[문제]
Q 2) 출력 채널별 L2 Norm 크기를 기준으로 하위 채널 전체를 프루닝하는 Channel-level Pruning을 구현합니다.

[요구사항]
- 각 출력 채널의 가중치 행렬에 대한 L2 Norm 계산 후 임계값 산출
- L2 Norm이 임계값 미만인 채널의 모든 가중치를 0으로 마스킹

In [ ]:
# [코드]
def prune_weight_channel_level(weight: torch.Tensor, sparsity: float) -> torch.Tensor:
    sparsity = min(1.0, max(0.0, sparsity))
    if sparsity == 1.0:
        weight.zero_()
        return torch.zeros_like(weight)
    elif sparsity == 0.0:
        return torch.ones_like(weight)

    num_channels = ### 작성 필요 ###
    num_pruned_channels = ### 작성 필요 ###
    importance = ### 작성 필요 ###

    threshold = torch.kthvalue(importance.flatten(), num_pruned_channels)[0]
    mask = importance > threshold
    mask = mask.expand_as(weight)
    weight.mul_(mask)
    return mask

## 문제 3. Kernel-level Pruning

[문제]
Q 3) 2D 합성곱 필터의 커널 단위(kernel_size x kernel_size) L2 Norm을 평가하여 필터를 마스킹하는 Kernel-level Pruning을 작성합니다.

[요구사항]
- 출력 채널과 입력 채널 축을 보존하고 커널 평면(H, W)에 대해 L2 Norm 계산
- 임계값 미만인 커널 전체를 0으로 마스킹

In [ ]:
# [코드]
def prune_weight_kernel_level(weight: torch.Tensor, sparsity: float) -> torch.Tensor:
    sparsity = min(1.0, max(0.0, sparsity))
    if sparsity == 1.0:
        weight.zero_()
        return torch.zeros_like(weight)
    elif sparsity == 0.0:
        return torch.ones_like(weight)

    num_kernels = ### 작성 필요 ###
    num_pruned_kernels = ### 작성 필요 ###
    importance = ### 작성 필요 ###

    threshold = torch.kthvalue(importance.flatten(), num_pruned_kernels)[0]
    mask = importance > threshold
    mask = mask.expand_as(weight)
    weight.mul_(mask)
    return mask

## 문제 4. Global Magnitude Pruning

[문제]
Q 4) 모델 내 모든 파라미터 텐서를 하나의 풀로 모아 전체 기준 하위 가중치를 프루닝하는 Global Magnitude Pruning을 구현합니다.

[요구사항]
- 프루닝 대상 파라미터들의 모든 원소를 1차원으로 펼쳐 결합 후 글로벌 임계값 산출
- 산출된 글로벌 임계값으로 각 레이어의 가중치에 마스크 적용

In [ ]:
# [코드]
def global_prune(model, sparsity):
    parameters_to_prune = []
    for name, param in model.named_parameters():
        if param.dim() > 1:
            parameters_to_prune.append(param.view(-1))

    all_weights = ### 작성 필요 ###
    num_elements = all_weights.numel()
    num_zeros = ### 작성 필요 ###
    importance = ### 작성 필요 ###
    threshold = ### 작성 필요 ###

    masks = {}
    for name, param in model.named_parameters():
        if param.dim() > 1:
            masks[name] = torch.abs(param.data) > threshold
    return masks

## 문제 5. LLM Magnitude Pruning

[문제]
Q 5) 대규모 언어 모델(LLM)의 선형 가중치 행렬들을 순회하며 글로벌 크기 기준 프루닝을 수행합니다.

[요구사항]
- nn.Linear 레이어의 가중치 텐서들을 식별하여 글로벌 임계값 산출 후 마스킹 적용

In [ ]:
# [코드]
@torch.no_grad()
def prune_magnitude(model, sparsity):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            num_elements = ### 작성 필요 ###
            num_zeros = ### 작성 필요 ###
            importance = ### 작성 필요 ###
            threshold = ### 작성 필요 ###
            mask = ### 작성 필요 ###
            W.mul_(mask)

## 문제 6. Wanda용 Activation L2 Norm 누적

[문제]
Q 6) 입력 활성화 텐서의 열별(채널별) 제곱합을 누적하여 활성화 L2 Norm 통계치를 갱신하는 헬퍼 함수를 작성합니다.

[요구사항]
- 배치 및 시퀀스 축에 대해 제곱합을 구하여 기존 통계 텐서에 누적

In [ ]:
# [코드]
def compute_activation_norm(x):
    x = x.type(torch.float32)
    activation_norm = ### 작성 필요 ###
    return activation_norm

## 문제 7. Wanda Pruning

[문제]
Q 7) 가중치 절댓값과 입력 활성화 L2 Norm의 곱으로 중요도를 평가하는 Wanda(Pruning by Weights and Activations) 알고리즘을 구현합니다.

[요구사항]
- 중요도 metric = |W| * ||X|| 산출
- 산출된 중요도의 하위 비율에 해당하는 가중치를 0으로 마스킹

In [ ]:
# [코드]
@torch.no_grad()
def prune_wanda(model, sparsity, input_feat):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            row, col = W.shape
            num_zeros_per_row = ### 작성 필요 ###
            importance = ### 작성 필요 ###
            threshold = ### 작성 필요 ###
            mask = ### 작성 필요 ###
            W.mul_(mask)

# Part 2. Quantization

모델을 layer 단위에서 양자화/역양자화 하는 과정을 다루는 문제입니다.

## 문제 8. Linear Quantization (FP → INT)

[문제]
Q 8) FP32 텐서를 scale과 zero_point를 이용해 양자화된 정수(INT) 텐서로 변환하는 Linear Quantization 함수를 구현합니다.

[요구사항]
- q = round(fp / scale) + zero_point 공식을 적용
- 양자화 비트 수(bitwidth)의 정수 유효 범위 [qmin, qmax]로 clamp 처리
- ※ 사전 정의 함수: get_quantized_range(bitwidth) -> (qmin, qmax) 가 제공됩니다.

In [ ]:
# [코드]
def linear_quantize(fp_tensor, bitwidth, scale, zero_point, dtype=torch.int8):
    scaled_tensor = ### 작성 필요 ###
    rounded_tensor = ### 작성 필요 ###
    rounded_tensor = rounded_tensor.to(dtype)
    shifted_tensor = ### 작성 필요 ###
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    quantized_tensor = shifted_tensor.clamp_(quantized_min, quantized_max)
    return quantized_tensor

## 문제 9. Quantization Scale & Zero Point 계산

[문제]
Q 9) FP32 텐서의 최솟값과 최댓값으로부터 균등 양자화 스케일(scale)과 제로 포인트(zero_point)를 산출합니다.

[요구사항]
- scale = (max_val - min_val) / (qmax - qmin) 산출
- zero_point = round(-min_val / scale) + qmin 계산 후 [qmin, qmax] 범위로 clamp
- ※ 사전 정의 함수: get_quantized_range(bitwidth) -> (qmin, qmax) 가 제공됩니다.

In [ ]:
# [코드]
def get_quantization_scale_and_zero_point(fp_tensor, bitwidth):
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    fp_max = fp_tensor.max().item()
    fp_min = fp_tensor.min().item()

    scale = ### 작성 필요 ###
    zero_point = ### 작성 필요 ###

    if zero_point < quantized_min:
        zero_point = quantized_min
    elif zero_point > quantized_max:
        zero_point = quantized_max
    else:
        zero_point = round(zero_point)
    return scale, int(zero_point)

## 문제 10. Quantized Linear: Output 재스케일 & Zero Point 보정

[문제]
Q 10) 정수 연산된 Quantized Linear의 출력 텐서에 대해 스케일 재조정 및 출력 제로 포인트를 보정합니다.

[요구사항]
- effective_scale = (input_scale * weight_scale) / output_scale 적용
- output_zero_point를 더하여 최종 INT8 출력 텐서 완성

In [ ]:
# [코드]
def quantized_linear_core(output, input_scale, weight_scale, output_scale, output_zero_point):
    weight_scale = weight_scale.flatten().view(1, -1)
    output = output.float()

    output = ### 작성 필요 ###
    output = ### 작성 필요 ###

    return output

## 문제 11. Quantized Conv2d: Output 재스케일 & Zero Point 보정

[문제]
Q 11) Quantized Conv2d 레이어의 출력 텐서에 입력/가중치/출력 스케일 팩터와 제로 포인트를 보정합니다.

[요구사항]
- 합성곱 누적 정수 결과에 스케일 비율을 곱하고 output_zero_point 보정

In [ ]:
# [코드]
def quantized_conv2d_core(output, input_scale, weight_scale, output_scale, output_zero_point):
    weight_scale = weight_scale.flatten().view(1, -1, 1, 1)
    output = output.float()

    output = ### 작성 필요 ###
    output = ### 작성 필요 ###

    return output

## 문제 12. K-means Quantization

[문제]
Q 12) K-means 클러스터링 알고리즘을 가중치 텐서에 적용하여 대표값 코드북(Codebook)을 학습하고 양자화 인덱스를 생성합니다.

[요구사항]
- 가중치 원소들을 1차원으로 변환하여 지정된 클러스터 수(2^bitwidth)로 KMeans 피팅
- 학습된 중심점(centroids)과 레이블(labels)을 Codebook 객체로 저장
- ※ 사전 정의 클래스: sklearn.cluster.KMeans 및 Codebook 객체가 제공됩니다.

In [ ]:
# [코드]
def k_means_quantize(fp32_tensor, bitwidth=4, codebook=None):
    if codebook is None:
        n_clusters = ### 작성 필요 ###
        kmeans = KMeans(n_clusters=n_clusters, mode='euclidean', verbose=0)
        labels = kmeans.fit_predict(fp32_tensor.view(-1, 1)).to(torch.long)
        centroids = kmeans.centroids.to(torch.float).view(-1)
        codebook = Codebook(centroids, labels)

    quantized_tensor = ### 작성 필요 ###
    fp32_tensor.set_(quantized_tensor.view_as(fp32_tensor))
    return codebook

## 문제 13. K-means Codebook 업데이트

[문제]
Q 13) K-means 양자화된 가중치에 대해 역전파 시 각 클러스터에 속한 가중치 기울기들의 합으로 코드북 중심점을 업데이트합니다.

[요구사항]
- 동일한 클러스터 인덱스를 갖는 가중치들의 gradient를 합산하여 centroid 갱신에 반영

In [ ]:
# [코드]
def update_codebook(fp32_tensor, codebook):
    n_clusters = codebook.centroids.numel()
    fp32_tensor = fp32_tensor.view(-1)
    for k in range(n_clusters):
        codebook.centroids[k] = ### 작성 필요 ###

# Part 3. Knowledge Distillation

지식증류의 원리를 이해하고, Teacher 모델의 예측값과 정답값을 활용해서 Student 모델 학습에 연결시키는 과정을 다루는 문제입니다.

## 문제 14. Knowledge Distillation (Soft Targets KL Loss)

[문제]
Q 14) Knowledge Distillation에서 Teacher 모델의 Soft Target과 Student의 출력 간 KL-Divergence 손실을 계산합니다.

[요구사항]
- 온도 파라미터(T)를 적용한 소프트맥스 확률 분포 생성
- F.kl_div를 계산하고 T^2 스케일 팩터를 곱하여 경사 하강 스케일 보정

In [ ]:
# [코드]
def kd_loss_step(teacher, student, inputs, labels, T,
                 soft_target_loss_weight, ce_loss_weight, ce_loss):
    with torch.no_grad():
        teacher_logits = teacher(inputs)
    student_logits = student(inputs)

    soft_targets = ### 작성 필요 ###
    student_prob = ### 작성 필요 ###

    soft_targets_loss = ### 작성 필요 ###

    label_loss = ce_loss(student_logits, labels)
    loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss
    return loss

## 문제 15. Cosine Loss 기반 KD

[문제]
Q 15) Teacher 모델과 Student 모델의 은닉 표현 벡터 간 코사인 유사도 손실(Cosine Loss)을 계산합니다.

[요구사항]
- 1 - cosine_similarity(student_rep, teacher_rep) 형태로 손실 계산하여 두 모델의 특징 표현 정렬

In [ ]:
# [코드]
def cosine_kd_step(teacher, student, inputs, labels,
                   hidden_rep_loss_weight, ce_loss_weight, ce_loss, cosine_loss):
    with torch.no_grad():
        _, teacher_hidden = teacher(inputs)
    student_logits, student_hidden = student(inputs)

    hidden_rep_loss = ### 작성 필요 ###

    label_loss = ce_loss(student_logits, labels)
    loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss

## 문제 16. MSE Feature Map 기반 KD

[문제]
Q 16) 중간 합성곱 피처맵의 형태를 일치시키고 MSE 손실을 기반으로 특징 맵 증류(Feature Map Distillation) 손실을 계산합니다.

[요구사항]
- Student 피처맵과 Teacher 피처맵의 요소별 평균 제곱 오차(F.mse_loss) 계산

In [ ]:
# [코드]
def mse_kd_step(teacher, student, inputs, labels,
                feature_map_weight, ce_loss_weight, ce_loss, mse_loss):
    with torch.no_grad():
        _, teacher_feature_map = teacher(inputs)
    student_logits, regressor_feature_map = student(inputs)

    hidden_rep_loss = ### 작성 필요 ###

    label_loss = ce_loss(student_logits, labels)
    loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss